# Genetic algorithms

In [1]:
using Distributions 
import Random: seed!
import LinearAlgebra: norm

## initialization

In [2]:
function rand_population_uniform(m, a, b)
    d = length(a)
    return [a + rand(d) .* (b - a) for i in 1:m]
end

function rand_population_normal(m, μ, Σ)
    D = MvNormal(μ, Σ)
    return [rand(D) for i in 1:m]

end

function rand_population_cauchy(m, μ, σ)
    n = length(μ)
    return [[rand(Cauchy(μ[j], σ[j])) for j in 1:n] for i in 1:m]
end

rand_population_binary(m, n) = [bitrand(n) for i in 1:m]

rand_population_binary (generic function with 1 method)

## selection

In [3]:
abstract type SelectionMethod end
struct TruncationSelection <: SelectionMethod
    k # top k to keep
end
function select(t::TruncationSelection, y)
    p = sortperm(y)
    return [p[rand(1:t.k, 2)] for i in y]

end
struct TournamentSelection <: SelectionMethod
    k
end
function select(t::TournamentSelection, y)
    getparent() = begin
        p = randperm(length(y))
        p[argmin(y[p[1:t.k]])]
    end
    return [[getparent(), getparent()] for i in y]
end
struct RouletteWheelSelection <: SelectionMethod end
function select(::RouletteWheelSelection, y)
    y = maximum(y) .- y
    cat = Categorical(normalize(y, 1))
    return [rand(cat, 2) for i in y]
end

select (generic function with 3 methods)

In [4]:


# Define probabilities for each category
probabilities = [0.1, 0.3, 0.6]

# Create a categorical distribution
cat = Categorical(probabilities)

# Sample from the categorical distribution
sample = rand(cat,10)
cat, sample

(Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.1, 0.3, 0.6]), [3, 2, 2, 1, 3, 3, 3, 1, 1, 3])

In [5]:
cat

Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.1, 0.3, 0.6])

## crossover

In [6]:
abstract type CrossoverMethod end
struct SinglePointCrossover <: CrossoverMethod end

function crossover(::SinglePointCrossover, a, b)
    i = rand(1:length(a))
    return vcat(a[1:i], b[i+1:end])

end
struct TwoPointCrossover <: CrossoverMethod end
function crossover(::TwoPointCrossover, a, b)
    n = length(a)
    i, j = rand(1:n, 2)
    if i > j
        (i, j) = (j, i)
    end
    return vcat(a[1:i], b[i+1:j], a[j+1:n])
end
struct UniformCrossover <: CrossoverMethod end
function crossover(::UniformCrossover, a, b)
    child = copy(a)
    for i in 1:length(a)
        if rand() < 0.5
            child[i] = b[i]
        end
    end
    return child
end

struct InterpolationCrossover <: CrossoverMethod
    λ
end

crossover(C::InterpolationCrossover, a, b) = (1 - C.λ) * a + C.λ * b

crossover (generic function with 4 methods)

## mutation

In [7]:
abstract type MutationMethod end
struct BitwiseMutation <: MutationMethod
    λ
end
function mutate(M::BitwiseMutation, child)
    return [rand() < M.λ ? !v : v for v in child]
end
struct GaussianMutation <: MutationMethod
    σ
end
function mutate(M::GaussianMutation, child)
    return child + randn(length(child)) * M.σ
end

mutate (generic function with 2 methods)

## GA

In [10]:
function genetic_algorithm(f, population, max_iter, selection, crossover_, mutation)
    m = length(population)
    n = length(population[1])
    y = [f(population[i]) for i in 1:m]
    for i in 1:max_iter
        parents = select(selection, y)
        
        children = [crossover(crossover_, population[p[1]], population[p[2]]) for p in parents]
        children = [mutate(mutation, c) for c in children]
        children_y = [f(c) for c in children]
        for j in 1:m
            if children_y[j] < y[j]
                y[j] = children_y[j]
                population[j] = children[j]
            end
        end
        # population = children
        # y = children_y
        @show minimum(y)
    end
    return population[argmin(y)]
    
end

genetic_algorithm (generic function with 1 method)

In [11]:

seed!(0) # set random seed for reproducible results
f = x -> norm(x)
m = 100 # population size
k_max = 100 # number of iterations
population = rand_population_uniform(m, [-300, -300], [300, 300])
S = TruncationSelection(10) # select top 10
C = SinglePointCrossover()
M = GaussianMutation(0.5) # small mutation rate
x = genetic_algorithm(f, population, k_max, S, C, M)
@show x

minimum(y) = 28.187204656076105
minimum(y) = 27.65186117747279
minimum(y) = 26.878825735583185
minimum(y) = 25.62665494435819
minimum(y) = 24.394822906725494
minimum(y) = 23.380636095768946
minimum(y) = 22.335950063225496
minimum(y) = 21.35383619168129
minimum(y) = 20.602741672359056
minimum(y) = 19.576559546891126
minimum(y) = 17.760488789107804
minimum(y) = 16.767634923776555
minimum(y) = 15.950584920471282
minimum(y) = 15.184344402731929
minimum(y) = 14.249021357683917
minimum(y) = 13.471475487688327
minimum(y) = 12.5556887636712
minimum(y) = 11.650167731242947
minimum(y) = 10.445343680507769
minimum(y) = 9.189168572077437
minimum(y) = 8.123911173826386
minimum(y) = 6.596026736429999
minimum(y) = 5.569042278265519
minimum(y) = 4.7084239097176885
minimum(y) = 3.8941381472617644
minimum(y) = 3.160249277326202
minimum(y) = 2.5588376803684616
minimum(y) = 1.4856592352347142
minimum(y) = 0.4369218680984593
minimum(y) = 0.05449192075204436
minimum(y) = 0.05449192075204436
minimum(y) = 0.0

2-element Vector{Float64}:
 -0.0004397464784187538
 -0.0032156600189333917

## TODO

1. Test the above implementation of GA on the Michalewicz function.

2. Write a function that runs the GA (above) for a `NRuns` number of times and saves the minimum of the function for each run in a vector. Report the average and standard deviation of the values. 

3. Test all the selection methods on the `norm` function in the example, using the function written at point 2. Is there any difference among results?

4. Choose the selection method you found to work best at 3. Test all recombination variants applicable. The same question.

5. Repeat for mutation.